# 🚬 煙霧偵測模型訓練 — Google Colab

**資料集：** [Roboflow smoke-gxoy3](https://universe.roboflow.com/naruesuan-university/smoke-gxoy3)  
**模型：** YOLOv8n（輕量，適合邊緣部署）  
**目標：** 訓練後匯出 ONNX 模型，複製到 `models/smoke_detector.onnx` 啟用條件 3

## 開始前請確認
- 上方選單：`執行階段 → 變更執行階段類型 → GPU (T4)`
- 準備好 Roboflow API Key（[取得方式](https://app.roboflow.com/) → Settings → Roboflow API）

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. 安裝套件

In [ ]:
!pip install -q roboflow ultralytics onnx onnxsim

## 2. 下載煙霧資料集

輸入你的 Roboflow API Key：

In [ ]:
from roboflow import Roboflow

API_KEY = "YOUR_ROBOFLOW_API_KEY"  # ← 替換為你的 Key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("naruesuan-university").project("smoke-gxoy3")

# 自動列出並使用最新可用版本（避免指定版本號不存在的問題）
versions = project.versions()
print(f"可用版本號：{[v.version for v in versions]}")
latest = versions[-1]
print(f"使用版本：{latest.version}")

dataset = latest.download("yolov8", location="/content/smoke_dataset")
print(f"資料集路徑：{dataset.location}")

## 3. 確認資料集結構與統計

In [ ]:
# 列出原始 data.yaml 供參考
!cat /content/smoke_dataset/data.yaml

## 4. 修正 data.yaml（確保路徑正確）

In [ ]:
import yaml, os, glob, shutil, random

base = "/content/smoke_dataset"

# 印出實際目錄結構
print("=== 實際目錄結構 ===")
for root, dirs, files in os.walk(base):
    depth = root.replace(base, "").count(os.sep)
    if depth > 2:
        continue
    print("  " * depth + os.path.basename(root) + "/")

# 偵測 train images
def find_images_dir(base, candidates):
    for c in candidates:
        p = os.path.join(base, c)
        imgs = glob.glob(f"{p}/*.jpg") + glob.glob(f"{p}/*.png") + glob.glob(f"{p}/*.jpeg")
        if imgs:
            return c, sorted(imgs)
    return None, []

train_rel, train_imgs = find_images_dir(base, ["train/images", "images/train", "train"])
val_rel,   val_imgs   = find_images_dir(base, ["valid/images", "val/images", "images/val", "valid", "val"])

assert train_rel and train_imgs, "❌ 找不到 train 影像目錄"

# 若沒有 val，自動從 train 切出 10%
if not val_rel:
    print(f"\n⚠️  未找到驗證集，從 train ({len(train_imgs)} 張) 自動切割 10% 作為 val ...")
    random.seed(42)
    random.shuffle(train_imgs)
    split = max(1, int(len(train_imgs) * 0.1))
    val_imgs_to_move   = train_imgs[:split]
    train_imgs_to_keep = train_imgs[split:]

    # 建立 val 目錄
    val_img_dir = os.path.join(base, "valid/images")
    val_lbl_dir = os.path.join(base, "valid/labels")
    os.makedirs(val_img_dir, exist_ok=True)
    os.makedirs(val_lbl_dir, exist_ok=True)

    train_lbl_dir = os.path.join(base, train_rel.replace("images", "labels"))

    for img_path in val_imgs_to_move:
        fname = os.path.basename(img_path)
        stem  = os.path.splitext(fname)[0]
        # 移動影像
        shutil.move(img_path, os.path.join(val_img_dir, fname))
        # 移動對應標籤（若存在）
        lbl_src = os.path.join(train_lbl_dir, stem + ".txt")
        if os.path.exists(lbl_src):
            shutil.move(lbl_src, os.path.join(val_lbl_dir, stem + ".txt"))

    val_rel = "valid/images"
    print(f"✅ 切割完成：train={len(train_imgs_to_keep)} 張，val={split} 張")
else:
    print(f"\n✅ train={len(train_imgs)} 張，val={len(val_imgs)} 張")

# 更新 data.yaml
yaml_path = f"{base}/data.yaml"
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg["path"]  = base
cfg["train"] = train_rel
cfg["val"]   = val_rel
cfg["nc"]    = 1
cfg["names"] = ["smoke"]

with open(yaml_path, "w") as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print("\n=== 修正後 data.yaml ===")
!cat /content/smoke_dataset/data.yaml

## 5. 訓練模型

T4 GPU 大約需要 **20–40 分鐘**（100 epochs, batch=16）

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print(f"使用裝置：{'GPU (' + torch.cuda.get_device_name(0) + ')' if device == 0 else 'CPU（建議切換至 GPU 執行階段）'}")

model = YOLO("yolov8s.pt")  # yolov8s：比 n 大 2 倍，準確度明顯提升

results = model.train(
    data="/content/smoke_dataset/data.yaml",
    epochs=100,
    batch=16 if device == 0 else 4,
    imgsz=640,
    device=device,
    project="/content/runs",
    name="smoke_detector",
    patience=20,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    save=True,
    save_period=10,
    exist_ok=True,
)

## 6. 查看訓練結果

In [ ]:
from IPython.display import Image as IPImage, display
import glob

# 訓練曲線
plots = glob.glob("/content/runs/smoke_detector/*.png")
for p in sorted(plots):
    display(IPImage(p))

In [ ]:
# 驗證最終指標
best_pt = "/content/runs/smoke_detector/weights/best.pt"
eval_model = YOLO(best_pt)
metrics = eval_model.val(data="/content/smoke_dataset/data.yaml")
print(f"mAP50   : {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall  : {metrics.box.mr:.4f}")

## 7. 匯出 ONNX 模型

In [ ]:
export_model = YOLO(best_pt)
export_model.export(
    format="onnx",
    imgsz=640,
    simplify=True,   # onnxsim 簡化計算圖，加速推論
    opset=17,
)

onnx_path = best_pt.replace(".pt", ".onnx")
print(f"\n✅ ONNX 模型位於：{onnx_path}")

## 8. 下載模型到本機

執行後會自動下載 `smoke_detector.onnx`，下載完成後複製到專案的 `models/` 目錄：

In [ ]:
import shutil
from google.colab import files

# 複製為統一命名
dst = "/content/smoke_detector.onnx"
shutil.copy(onnx_path, dst)
print(f"檔案大小：{os.path.getsize(dst)/1024/1024:.1f} MB")

# 下載到本機
files.download(dst)
print("\n📦 下載完成後，將 smoke_detector.onnx 複製到專案的 models/ 目錄")
print("   路徑：cgr_detection/models/smoke_detector.onnx")
print("   重啟推論系統即自動啟用條件 3（煙霧偵測）")

## 9. （選用）同時下載 .pt 權重備份

In [ ]:
# 如果想保留 PyTorch 權重以便日後繼續訓練
files.download(best_pt)

---

## 完成後的部署步驟

```bash
# 1. 將下載的模型放到專案目錄
cp ~/Downloads/smoke_detector.onnx /path/to/cgr_detection/models/

# 2. 重啟推論系統（自動偵測並啟用條件 3）
python infer_main.py
```

啟動時會看到：
```
[func] 煙霧偵測模型已啟用（條件 3）
```

條件 3 觸發時畫面顯示橘色煙霧框與 `[+10 Smoke]` 計分提示。